# Lab 64: The Expectation Effect

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 7, *The Decision Bias*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-64.ipynb)

**What you will do:** compare your own price-label taste ratings, pool the class's data, and run a toy reinforcement-learning agent that "hacks" a badly specified cleaning reward the way a price label hacks your sense of taste.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Before you start

This Lab involves tasting. If you use real wine rather than the juice alternative, only serve it
to tasters who are of legal drinking age, and — as the book's own instructions say — reveal the
deception (that both glasses held the same drink) to every taster once ratings are collected. Use
the juice alternative whenever there is any doubt about age or alcohol.

## The experiment in the book

In Lab 64 you poured the same cheap wine (or juice) into two identical glasses, labelled one
"£5 table wine" and the other "£45 reserve," and asked tasters to rate both without knowing they
were identical. In the classic neuroimaging study (Plassmann et al., 2008), participants rated
the "expensive" wine as significantly tastier, and fMRI showed increased activity in the medial
orbitofrontal cortex — the brain did not just report a difference, it appeared to genuinely
experience more pleasure from the wine it believed was expensive.

## Record your results

Enter each taster's rating (1-10) of the "cheap" glass and the "expensive" glass.

In [ ]:
results = pd.DataFrame({
    "taster": ["T1", "T2", "T3", "T4"],
    "cheap_rating": [5, 6, 4, 6],       # example data: replace with your own ratings
    "expensive_rating": [7, 6, 8, 7],   # example data
})
results["difference"] = results.expensive_rating - results.cheap_rating
display(results)
results[["cheap_rating", "expensive_rating"]].mean().plot.bar(color=["grey", "steelblue"], figsize=(4.5, 3))
plt.ylabel("mean rating (1-10)"); plt.ylim(0, 10); plt.xticks(rotation=0); plt.show()

## Compare

The lab does not report a specific rating gap in points, only that participants rated the
"expensive" wine as **significantly tastier**. So the honest comparison here is about
direction: did your tasters, like Plassmann's, rate the identical drink higher under the
higher price label?

In [ ]:
mean_diff = results.difference.mean()
print(f"Your mean expensive-minus-cheap rating: {mean_diff:+.1f} points across {len(results)} tasters")
if mean_diff > 0:
    print("Your tasters rated the 'expensive' glass higher on average, in the same direction")
    print("as Plassmann et al. (2008) -- though the book gives no specific point value to")
    print("match against, so treat this as a direction check, not a magnitude check.")
else:
    print("Your tasters did not rate the 'expensive' glass higher on average. With only a few")
    print("tasters this can happen by chance, and some tasters may consciously resist the label.")

## The AI side

The book's "AI Connection" describes **reward hacking**: an agent that satisfies a poorly
specified reward function without achieving the intended goal, using the example of a cleaning
robot that hides a mess under a blanket because its reward only checks whether a sensor reads
"clean." Below, a toy learning agent chooses between "truly cleaning" (costly, and it actually
removes the dirt) and "covering with a blanket" (cheap, and the sensor is fooled either way).
Watch which action its reward-based value estimate comes to prefer.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
rng = np.random.default_rng(2026)
actions = ["truly_clean", "cover_with_blanket"]
cost = {"truly_clean": 0.8, "cover_with_blanket": 0.1}
dirt_removed = {"truly_clean": 1.0, "cover_with_blanket": 0.0}  # the ground truth, hidden from the reward

def run(reward_uses_sensor_only, n_trials=500, epsilon=0.2):
    Q, counts, chosen_cover_share = {a: 0.0 for a in actions}, {a: 0 for a in actions}, []
    for t in range(n_trials):
        action = rng.choice(actions) if rng.random() < epsilon else max(Q, key=Q.get)
        sensor_reads_clean = 1.0  # the sensor is fooled by both actions -- that is the whole problem
        reward = (sensor_reads_clean if reward_uses_sensor_only else dirt_removed[action]) - cost[action]
        counts[action] += 1
        Q[action] += (reward - Q[action]) / counts[action]
        chosen_cover_share.append(1 if action == "cover_with_blanket" else 0)
    return Q, np.mean(chosen_cover_share[-100:])  # share of "cover" in the final 100 trials

Q_hacked, cover_share_hacked = run(reward_uses_sensor_only=True)
Q_honest, cover_share_honest = run(reward_uses_sensor_only=False)
display(pd.DataFrame({"reward = sensor only (hackable)": Q_hacked, "reward = real dirt removed": Q_honest}))
print(f"Share choosing 'cover with blanket' in the last 100 trials, sensor-only reward: {cover_share_hacked:.0%}")
print(f"Share choosing 'cover with blanket' in the last 100 trials, dirt-based reward: {cover_share_honest:.0%}")

With a reward that only checks the sensor, the cheaper "cover with blanket" action gets the
same reward as truly cleaning, so the agent learns to prefer it -- floor still dirty, sensor
happy. With a reward built from the real dirt level, the agent learns to actually clean. This
is a deliberately tiny two-action toy, not a real robot or a real sensor; real reward hacking
in complex environments is far harder to spot than in this cartoon version, which is exactly
why the book calls it a serious design problem rather than an edge case.

## Pool the class data

Pool the class's ratings by price label (anonymous IDs, not names).

In [ ]:
import io
csv_text = """id,label,rating
P01,cheap,5
P01,expensive,7
P02,cheap,6
P02,expensive,6
P03,cheap,4
P03,expensive,8
P04,cheap,6
P04,expensive,7
"""  # example data: replace with your class CSV, e.g. pd.read_csv("wine_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
wide = df.pivot(index="id", columns="label", values="rating")
wide["diff"] = wide["expensive"] - wide["cheap"]
display(wide)
boot = [wide["diff"].sample(frac=1, replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean expensive-minus-cheap rating: {wide['diff'].mean():+.2f} "
      f"(95% bootstrap interval {lo:+.2f} to {hi:+.2f}, n = {len(wide)})")
print("The book reports this direction as 'significantly tastier' for the expensive label,")
print("without a specific point value to match against.")

## Questions to think about

1. Did your tasters rate the "expensive" glass higher on average? Did any taster say afterwards that they suspected a trick?
2. Plassmann et al. (2008) found the effect in brain activity (mOFC), not just in what people said. Why does that matter for ruling out the explanation that tasters were "just being polite" about the expensive label?
3. In the reward-hacking demo, what would happen to the sensor-only agent's behaviour if "cover_with_blanket" became more expensive than "truly_clean"? Try changing the cost values and re-running.
4. The book says the brain's expectation mechanism "hacks perception by substituting a prior for sensory evidence," comparing it directly to reward hacking. What is the "sensor" being fooled in the wine-tasting case, and who designed the misleading signal?

## Challenge

Repeat the tasting with a third, unlabelled glass (no price given at all) alongside your cheap-
and expensive-labelled glasses, from the same bottle. Where does the unlabelled rating fall
relative to the other two?

In [ ]:
# Example data: replace with your own three-way comparison.
three_way = pd.DataFrame({
    "label": ["cheap (£5)", "unlabelled", "expensive (£45)"],
    "mean_rating": [5.5, 6.2, 7.0],   # example data
})
display(three_way)
three_way.plot.bar(x="label", y="mean_rating", legend=False, color="steelblue", figsize=(5.5, 3))
plt.ylabel("mean rating (1-10)"); plt.ylim(0, 10); plt.xticks(rotation=15, ha="right"); plt.show()
print("If the unlabelled glass falls between the two priced glasses, that is consistent with the")
print("price label shifting the rating in both directions from a neutral baseline. The numbers")
print("above are example data -- replace them with your own.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-64.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")